#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
from pyspark.sql.functions import regexp_replace, replace

# Read Bronze table

In [0]:
df = spark.table("`abhi-de-ete-project-catlog`.bronze.crm_cust_info")

#Silver Transformations

##Trimming

In [0]:
df.printSchema()

In [0]:
df.display(5)

In [0]:
df.schema.fields

In [0]:
print(df.schema.fields[0].name)
print(df.schema.fields[0].dataType)

In [0]:
# From Above cell there seems to be spaces in the cell, 
# Lets trim all string columns

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Normalization

In [0]:
#Get the distinct values of the column
df.select("cst_marital_status").distinct().show()
df.select("cst_gndr").distinct().show()

In [0]:
df = (
    df.
        withColumn("cst_marital_status", regexp_replace(col("cst_marital_status"), "M", "Married")).
        withColumn("cst_marital_status", regexp_replace(col("cst_marital_status"), "S", "Single"))
    )  

In [0]:
df.show(5)

In [0]:

df = (
    df
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

In [0]:
df.show(5)

##Remove Records with Missing Customer ID

In [0]:
df.filter(df.cst_id.isNull()).show()

In [0]:
# It seems there is a null value in the id column, lets remove it.
df = df.filter(col("cst_id").isNotNull())

In [0]:
df.filter(df.cst_id.isNull()).show()

## Renaming Columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.columns

## Sanity checks of dataframe

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
#Create the slver schema with if not exists logic 

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `abhi-de-ete-project-catlog`.silver")

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("`abhi-de-ete-project-catlog`.silver.crm_customers")

## Sanity checks of silver table

In [0]:
#import the method check_df_sanity from the fpython file located at same file folder stucture from file sanity_check_for_dataframe
from sanity_check_for_dataframe import check_df_sanity
check_df_sanity(df)



In [0]:
%sql
SELECT * FROM `abhi-de-ete-project-catlog`.silver.crm_customers LIMIT 10